# Smart 5-Fold Cross Validation Setup
This notebook creates a Multilabel Stratified K-Fold split for your YOLO dataset.
It prevents data-leakage by splitting on **original base images**, and then grouping their respective `_px.jpg` patches into the assigned train or validation fold.

**Note for Kaggle / Google Colab Users:**
Make sure to update the `DATASET_BASE_PATH` variable in the execution cell so that it points to your Kaggle or Colab dataset path, e.g.:
- Kaggle: `/kaggle/input/your-dataset/YOLO_Sahi_Dataset`
- Colab: `/content/drive/MyDrive/YOLO_Sahi_Dataset`


In [ ]:
!pip install iterative-stratification pyyaml

In [ ]:
import os
import json
import glob
import numpy as np
from collections import defaultdict
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import yaml

# --- CONFIGURATION ---
# Base path of your YOLO dataset output
DATASET_BASE_PATH = "/Users/sahindogruca/Desktop/code/ytü/semester-8/OxidativeStress/YOLO_Sahi_Dataset"

# Where original base JSONs are located (for extracting base class labels)
JSON_DIR = "/Users/sahindogruca/Desktop/code/ytü/semester-8/OxidativeStress/OksidatifStress/json_files"

# Output directory for the new K-Fold files inside the dataset
KFOLD_OUT_DIR = os.path.join(DATASET_BASE_PATH, "kfold_yamls_smart")

# Class dictionary mappings for your dataset
CLASS_MAP = {
    "DEG": 0,
    "NH":  1,
    "SH":  2,
    "MH":  3,
    "BH":  4
}

os.makedirs(KFOLD_OUT_DIR, exist_ok=True)

In [ ]:
print("Parsing Original JSON files to calculate base image stratification...")

base_image_names = []
base_image_labels = []

json_files = glob.glob(os.path.join(JSON_DIR, "*.json"))

for jf in json_files:
    basename = os.path.splitext(os.path.basename(jf))[0]
    
    with open(jf, 'r') as f:
        data = json.load(f)
        
    counts = np.zeros(len(CLASS_MAP))
    for shape in data.get("shapes", []):
        lbl = shape.get("label", "")
        if lbl in CLASS_MAP:
            counts[CLASS_MAP[lbl]] += 1
            
    # Binarize label presence for iterative stratification 
    # (Does this original image contain at least one of this class?)
    presence = (counts > 0).astype(int)
    
    base_image_names.append(basename)
    base_image_labels.append(presence)

base_image_names = np.array(base_image_names)
base_image_labels = np.array(base_image_labels)

print(f"Parsed {len(base_image_names)} original base images.")

In [ ]:
print("Performing Multilabel Stratified 5-Fold Split...")
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

yolo_images_dir = os.path.join(DATASET_BASE_PATH, "images")
all_patches = glob.glob(os.path.join(yolo_images_dir, "*_p*.jpg"))
print(f"Found {len(all_patches)} total patches in the YOLO dataset.")

# Create mappings from base image to its patches
base_to_patches = defaultdict(list)
for p in all_patches:
    filename = os.path.basename(p)
    name_no_ext = os.path.splitext(filename)[0]
    
    # Find last '_p' occurrence to get base name (e.g. Hidayet26_p0 -> Hidayet26)
    idx = name_no_ext.rfind("_p")
    if idx != -1:
        base = name_no_ext[:idx]
    else:
        # fallback just in case there's no _p suffix
        base = name_no_ext
        
    base_to_patches[base].append(filename)

# Dummy X as iterative stratification only cares about y for stratification setup
X = np.zeros((len(base_image_names), 1))

for fold, (train_idx, val_idx) in enumerate(mskf.split(X, base_image_labels)):
    print(f"\nProcessing Fold {fold}...")
    
    train_bases = base_image_names[train_idx]
    val_bases = base_image_names[val_idx]
    
    train_txt_path = os.path.join(KFOLD_OUT_DIR, f"smart_fold_{fold}_train.txt")
    val_txt_path = os.path.join(KFOLD_OUT_DIR, f"smart_fold_{fold}_val.txt")
    
    # By writing relative paths e.g., './images/Hidayet26_p0.jpg',
    # they automatically resolve to where the yaml 'path:' specifies.
    train_count = 0
    with open(train_txt_path, "w") as f:
        for base in train_bases:
            patches = base_to_patches.get(base, [])
            for p in patches:
                f.write(f"./images/{p}\n")
                train_count += 1
                
    val_count = 0
    with open(val_txt_path, "w") as f:
        for base in val_bases:
            patches = base_to_patches.get(base, [])
            for p in patches:
                f.write(f"./images/{p}\n")
                val_count += 1
                
    print(f"Fold {fold} | Train patches: {train_count} | Val patches: {val_count}")
    
    # Generate the YAML file containing Kaggle/Colab relative compatible paths
    yaml_path = os.path.join(KFOLD_OUT_DIR, f"smart_fold_{fold}.yaml")
    
    yaml_content = {
        "path": DATASET_BASE_PATH,
        "train": f"kfold_yamls_smart/smart_fold_{fold}_train.txt",
        "val": f"kfold_yamls_smart/smart_fold_{fold}_val.txt",
        "names": CLASS_MAP
    }
    
    # Invert the CLASS_MAP for the YOLO yaml structure (YOLO wants int -> name)
    inv_class_map = {v: k for k, v in CLASS_MAP.items()}
    yaml_content["names"] = inv_class_map
    
    with open(yaml_path, "w") as yaml_file:
        yaml.dump(yaml_content, yaml_file, sort_keys=False)

print("\nAll folds generated successfully!")